In [1]:
import findspark
findspark.init()
from pyspark import SparkConf, SparkContext
from pyspark.sql.types import StringType
from pyspark import SQLContext

conf = SparkConf().setMaster('local').setAppName('Mi programa')
sc = SparkContext(conf = conf)
sqlContext = SQLContext(sc)

In [2]:
lines = sc.textFile('ejemplopyspark.txt')
#lines = lines.repartition(3)
lines.getNumPartitions()

1

In [21]:
py = sc.accumulator(0)
sp = sc.accumulator(0)

def lenguajes(linea):
    global py, sp
    if 'Python' in linea:
        py += 1
        if 'Aprendizaje' in linea:
            sp += 1
        return True   
    elif 'Aprendizaje' in linea:
        sp += 1
        return True
    else:
        return False
    
valores = lines.filter(lenguajes)    

In [22]:
valores.count()

21

In [33]:
py

Accumulator<id=8, value=6>

In [32]:
sp

Accumulator<id=9, value=36>

In [24]:
functionmap = valores.map(lambda x: (x,1))

In [25]:
contarvalores = functionmap.reduceByKey(lambda x,y: x+y)

In [26]:
contarvalores.count()

20

In [27]:
contarvalores.sortBy(lambda x: x[1],ascending=False).take(5)

[('Aprendizaje automático', 2),
 ('    5 Distinción entre Aprendizaje supervisado y no supervisado', 1),
 ('Los diferentes algoritmos de Aprendizaje Automático se agrupan en una taxonomía en función de la salida de los mismos. Algunos tipos de algoritmos son:',
  1),
 ('Aprendizaje supervisado', 1),
 ('Artículo principal: Aprendizaje supervisado', 1)]

In [28]:
def lenguajes_map(x):
    if 'Python' in x and 'Aprendizaje' in x:
        return ('Count',(1,1))
    elif 'Python' in x:
        return ('Count',(1,0))
    elif 'Aprendizaje' in x:
        return ('Count',(0,1))
    else:
        return ('Count',(0,0))
    
mapfun = lines.map(lenguajes_map)

In [29]:
mapfun.count()

366

In [30]:
lines.count()

366

In [31]:
# Número de veces que aparece 'Pyhton' y numero de veces que aparece 'Aprendizaje'

mapfun.reduceByKey(lambda x,y: (x[0]+y[0], x[1]+y[1])).collect()

[('Count', (3, 18))]